# Calculate sectors footprint with EXIOBASE

## Notebook README

This work is licenced under the Creative Commons Attribution (CC-BY 4.0) public licence.

**Related publication**

If you utilize any portions of the code, results, or draw inspiration for your projects, please reference the published article below.

 - Title: Residual biomass to bio-based chemicals and plastics: ex-ante screening methodology for prioritizing high-impact substitutions
 - Authors: [Nicolas LIENART](https://orcid.org/0009-0001-3259-2819), [Thibaut LECOMPTE](https://orcid.org/0000-0001-9237-8454), [Lorie HAMELIN](https://orcid.org/0000-0001-9092-1900) 
 - Journal: Resources, Conservation and Recycling (RCR) - Elsevier
 - Doi: #todo
 - Git repository (Forge INRAE): https://forge.inrae.fr/nicolas.lienart/screen-lca-paper-supplementary-code
 - Git repository (GitHub): https://github.com/nicolnt/screen-lca-paper-supplementary-code

**Description and details**

This code calculates the greenhouse gas (GHG) footprint of a given list of regions (e.g., single country like FR, a region like EU27) imports and production for a given year (e.g., 2019, 2022) based on EXIOBASE data.
The GHG indicator is the GWP100 calculated using coefficients from IPCC AR6 2021.
Refer to the aforementioned main manuscript and accompanying supplementary information documents for more details.

**Updates**

 - May 04, 2026: Ready for submission
 - August 06, 2026: Updated variable names and clarified some functions. Reorganized the heading structure for consistency.
 - August 07, 2026: Extensively cleaned the file: improved characterization data quality, use official characterization method, made the code region agnostic to permit sensitivity analysis, bump EXIOBASE version to v3.10.2 (published May 13, 2026).

**Package versions**

 - See [`environment.yml`](./environment.yml)
 - pymrio library code fetched directly from GitHub. See: https://github.com/IndEcol/pymrio/issues/144

 - Exiobase version: 3.10.2
   - Available at https://zenodo.org/records/20051562
   - datasets:
     - IOT_2019_pxp - https://zenodo.org/records/20051562/files/IOT_2019_pxp.zip?download=1
     - IOT_2022_pxp - https://zenodo.org/records/20051562/files/IOT_2022_pxp.zip?download=1

**Relevant references**

 - Pellan, M. et al. (2024) “Integrating Consumption-Based Metrics into Sectoral Carbon Budgets to Enhance Sustainability Monitoring of Building Activities,” Sustainability, 16(16), p. 6762. Available at: https://doi.org/10.3390/su16166762.
   - _Provided insiration on the use of Pymrio with Exiobase, see associated Github repository_
 - Aguilar-Hernandez, G.A. (2025) “A Novel Framework to Measure Circularity Trade-offs and Synergies in the Global Context,” Journal of Circular Economy, 3(1). Available at: https://doi.org/10.55845/YTGD9041.
   - _Provided the Python code for GHG stressors characterization, see associated Github repository_
 - Andrieu, B. et al. (2024) “An open-access web application to visualise countries’ and regions’ carbon footprints using Sankey diagrams,” Communications Earth & Environment, 5(1), pp. 1–9. Available at: https://doi.org/10.1038/s43247-024-01378-8.
   - _Provided insiration on the use of Pymrio with Exiobase, see associated Github repository_
 - Wiedmann, T. (2017) “On the decomposition of total impact multipliers in a supply and use framework,” Journal of Economic Structures, 6(1), p. 11. Available at: https://doi.org/10.1186/s40008-017-0072-0.
   - _Provided example of input-output calculations and results_
 - Stadler, K. (2021) “Pymrio – A Python Based Multi-Regional Input-Output Analysis Toolbox,” Journal of Open Research Software, 9(1). Available at: https://doi.org/10.5334/jors.251.
   - _Provided example of input-output calculations and results_
 - https://pymrio.readthedocs.io/en/latest/
   - _Pymrio library documentation_

**Exiobase Copyright Notice**

EXIOBASE is managed by the EXIOBASE consortium. 

Copyright holders
- MSUT, IOTs, emission data: © 2024, XIO SA AS  
- Material extensions: © 2024, Vienna University of Economics and Business (WU) 
- The material extensions are released under a CC-BY-SA license instead of the license conditions above.

## Initialization

In [1]:
import pymrio
import numpy as np
import pandas as pd
from datetime import datetime
import country_converter as coco

In [2]:
print('numpy version:', np.__version__)
print('pandas version:', pd.__version__)
print('pymrio version:', pymrio.__version__)

numpy version: 2.4.3
pandas version: 2.2.3
pymrio version: 0.6.3


In [3]:
OUTPUT_PATH = 'output'
IO_DATA_PATH = './Exiobase_data/'
EXIOBASE_VERSION = 'EXIOBASE_v3.10.2'
EXIOBASE_MODEL_AND_YEAR = 'IOT_2022_pxp'
EXIOBASE_PATH = IO_DATA_PATH + EXIOBASE_VERSION + '/' + EXIOBASE_MODEL_AND_YEAR
# VALUE_COLUMN_NAME = 'value (kg CO2 eq.)'
# SELECTED_REGION = 'FR' # Use France as study region
SELECTED_REGION = 'EU27' # Use European Union 27 (2020) as study region

print(EXIOBASE_PATH)

./Exiobase_data/EXIOBASE_v3.10.2/IOT_2022_pxp


In [4]:
# NOTE: (re)Load Exiobase data

if 'exiobase' in globals():
    del exiobase

# Takes < 1 min
exiobase = pymrio.parse_exiobase3(EXIOBASE_PATH)

In [5]:
# NOTE: Calculate missing matrices, required for later calculations

# Can take a few minutes
# exiobase.calc_all()

# Much faster (< 1 min)
if not hasattr(exiobase, 'A') or exiobase.A is None: exiobase.A = pymrio.calc_A(exiobase.Z, exiobase.x)
if not hasattr(exiobase, 'L') or exiobase.L is None: exiobase.L = pymrio.calc_L(exiobase.A)

In [6]:
# NOTE: Export function (optional)
def export_csv(exiobase, data, name: str):
    date = datetime.now().strftime('%Y%m%d')
    data.to_csv(f"{OUTPUT_PATH}/{date} - output_exiobase_v{exiobase.meta.version}_{exiobase.meta.name}_{name}_{SELECTED_REGION}.csv")

## Stressors characterization

Manual characterization originally inspired from https://github.com/aguilarga/circularity_trade-offs-synergies_supplementary_material/blob/main/ghg_calculation_exiobase_v3.9.5.py
Linked to this publication: Aguilar-Hernandez, G.A. (2025) “A Novel Framework to Measure Circularity Trade-offs and Synergies in the Global Context,” Journal of Circular Economy, 3(1). Available at: https://doi.org/10.55845/YTGD9041.

However, some flows where missing in this work and added.
All the cheracterization factor were taken from IPCC 2021 method, obtained with the help of Brightway.

Brightway method key: `('ecoinvent-3.11', 'IPCC 2021', 'climate change: total (excl. biogenic CO2)', 'global warming potential (GWP100)')`

Now following the official `pymrio` method to characterize stressors: https://pymrio.readthedocs.io/en/latest/notebooks/stressor_characterization.html

In [7]:
gwp_characterization_table = pd.read_csv('IPCC 2021 GWP100 characterization.csv')

In [8]:
characterized_air_emissions = exiobase.air_emissions.characterize(gwp_characterization_table)

In [9]:
exiobase.ghg_impacts = characterized_air_emissions.extension

In [10]:
# NOTE: Calculate all matrices (incl. L, S, D_cba, D_pba), including those related to the characterized impacts

# Can take a few minutes
# exiobase.calc_all()

# Much faster (< 1 min)
exiobase.ghg_impacts.calc_system(exiobase.x, exiobase.Y, L=exiobase.L)

In [11]:
gwp_unit = characterized_air_emissions.extension.unit.loc[:, 'unit'].values[0]

## Regions

In [12]:
cc = coco.CountryConverter()
EU27_ISO2_list = cc.EU27as('ISO2')['ISO2'].to_list()
# EU27_ISO2_list

In [13]:
regions = {
    'FR': ['FR'],
    'EU27': EU27_ISO2_list
}

## Accounting approach: Footprint of production + imports (as used in the main paper case study)

### 1.1 Imported final demand footprint in the given region per product

In [14]:
Y_imports = exiobase.Y.loc[:, regions[SELECTED_REGION]]

Y_imports.loc[regions[SELECTED_REGION], regions[SELECTED_REGION]] = 0

In [15]:
Y_imports = exiobase.Y.loc[:, regions[SELECTED_REGION]]
#specific_index_columns

# NOTE: Setting the given region final demand satisfied by the given region to 0. Avoiding double counting with the given regionR production footprint
Y_imports.loc[regions[SELECTED_REGION], regions[SELECTED_REGION]] = 0

# NOTE: Join the different final demand category columns together, no distinction is made
Y_imports = Y_imports.sum(1)

diag_Y_imports = pd.DataFrame(np.diag(Y_imports))
diag_Y_imports.index = Y_imports.index 
diag_Y_imports.columns = Y_imports.index

In [16]:
# NOTE: Multiply the given region final demand with the footprint data
Y_imports_footprint = exiobase.ghg_impacts.M.dot(diag_Y_imports)

In [17]:
Y_imports_footprint_agg = Y_imports_footprint.T.groupby('sector').sum()
Y_imports_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,4.056150e+07
Air transport services (62),3.959442e+10
Aluminium and aluminium products,1.436627e+09
Aluminium ores and concentrates,4.890810e+05
Animal products nec,1.375600e+09
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,3.436975e+05
Wood waste for treatment: landfill,9.065225e+04


In [18]:
# NOTE: Export restults to CSV
export_csv(exiobase, Y_imports_footprint_agg, "Y_imports_footprint")

### 1.2 Imported intermediate demand footprint in the given region per product

In [19]:
# NOTE: Only conserve intermediate industry demand from the given region
Z_imports = exiobase.Z.loc[:, regions[SELECTED_REGION]]

# NOTE: Sum all intermediate demand categories (columns) together,
# no distinction is made regarding the origin of the industry
Z_imports = Z_imports.sum(1)

# NOTE: Set locally satisfied intermediate demand from the given region to 0.
# Avoiding double counting with the given region production footprint
Z_imports.loc[regions[SELECTED_REGION]] = 0

diag_Z_imports = pd.DataFrame(np.diag(Z_imports))
diag_Z_imports.index = Z_imports.index 
diag_Z_imports.columns = Z_imports.index

In [20]:
# NOTE: Multiply the given region intermediate demand with the footprint data

Z_imports_footprint = exiobase.ghg_impacts.M.dot(diag_Z_imports)

In [21]:
Z_imports_footprint_agg = Z_imports_footprint.T.groupby('sector').sum()
Z_imports_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,8.494636e+07
Air transport services (62),3.140665e+10
Aluminium and aluminium products,2.289059e+10
Aluminium ores and concentrates,1.371586e+07
Animal products nec,2.961956e+08
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,2.826727e+06
Wood waste for treatment: landfill,5.443877e+05


In [22]:
export_csv(exiobase, Z_imports_footprint_agg, "Z_imports_footprint")

### 2.1 Production footprint in the given region per product, for final demand (the given region and exports)

In [23]:
Y_production = exiobase.Y.copy()

# NOTE: all other producers in Z than the given region are set to 0, we only consider production happening in the given region
Y_production.loc[~Y_production.index.get_level_values('region').isin(regions[SELECTED_REGION])] = 0

Y_production

# NOTE: Join the different final demand category columns together, no distinction is made
Y_production = Y_production.sum(1)


diag_Y_production = pd.DataFrame(np.diag(Y_production))
diag_Y_production.index = Y_production.index 
diag_Y_production.columns = Y_production.index

In [24]:
# NOTE: Multiply the given region production feeding different final demand with the footprint data

Y_production_footprint = exiobase.ghg_impacts.M.dot(diag_Y_production)

In [25]:
Y_production_footprint_agg = Y_production_footprint.T.groupby('sector').sum()
Y_production_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,6.352942e+07
Air transport services (62),7.094385e+10
Aluminium and aluminium products,3.107898e+09
Aluminium ores and concentrates,1.267618e+06
Animal products nec,3.297590e+09
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,1.203739e+09
Wood waste for treatment: landfill,4.403761e+08


In [26]:
export_csv(exiobase, Y_production_footprint_agg, "Y_production_footprint")

### 2.2 Production footprint in the given region per product, for intermediate demand (the given region and exports)

In [27]:
Z_production = exiobase.Z.copy()

# NOTE: all other producers in Z than the given region are set to 0, we only consider production happening in the given region
Z_production.loc[~Y_production.index.get_level_values('region').isin(regions[SELECTED_REGION])] = 0


# NOTE: Only conserve industry production from the given region.
Z_production

# NOTE: Sum all output categories (columns) together,
# No distinction is made regarding the demanding industry
Z_production = Z_production.sum(1)
Z_production

# NOTE: Diagonalized Z (gross output) where only the given region output is kept
diag_Z_production = pd.DataFrame(np.diag(Z_production))
diag_Z_production.index = Z_production.index 
diag_Z_production.columns = Z_production.index

In [28]:
# NOTE: Multiply the given region production with the footprint data.

Z_production_footprint = exiobase.ghg_impacts.M.dot(diag_Z_production)

In [29]:
Z_production_footprint_agg = Z_production_footprint.T.groupby('sector').sum()
Z_production_footprint_agg

impact,climate change: total (excl. biogenic CO2)
sector,
Additives/Blending Components,1.125740e+09
Air transport services (62),5.563348e+10
Aluminium and aluminium products,4.788553e+10
Aluminium ores and concentrates,3.770405e+08
Animal products nec,1.220785e+09
...,...
"Wood material for treatment, Re-processing of secondary wood material into new wood material",0.000000e+00
Wood waste for treatment: incineration,3.787428e+09
Wood waste for treatment: landfill,2.658202e+09


In [30]:
export_csv(exiobase, Z_production_footprint_agg, "Z_production_footprint")

## Total footprint

### Consumption footprint of the given region

In [31]:
D_cba = exiobase.ghg_impacts.D_cba_reg.loc[:, regions[SELECTED_REGION]]
f"{D_cba.iloc[0].sum():.2E} {gwp_unit}"

'4.44E+12 kg CO2-Eq'

### Production footprint of the given region

In [32]:
D_pba = exiobase.ghg_impacts.D_pba_reg.loc[:, regions[SELECTED_REGION]]
f"{D_pba.iloc[0].sum():.2E} {gwp_unit}"

'3.36E+12 kg CO2-Eq'

### Direct emissions of the final demand of the given region

In [33]:
F_Y = exiobase.ghg_impacts.F_Y.loc[:, regions[SELECTED_REGION]]
f"{F_Y.iloc[0].sum():.2E} {gwp_unit}"

'7.42E+11 kg CO2-Eq'